# SCDisrupt-Synth v1.0 Benchmark Notebook

This notebook provides a simple, reproducible baseline for the **SCDisrupt-Synth v1.0** dataset.

**Dataset DOI:** https://doi.org/10.5281/zenodo.22073886

It shows how to:
- load the dataset,
- inspect class balance and source types,
- train a text-classification baseline,
- report accuracy, precision, recall, and F1,
- compare performance across source types.

> Important: SCDisrupt-Synth is fully synthetic. Benchmark results should not be interpreted as real-world operational performance.


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

pd.set_option("display.max_colwidth", 120)


## 2. Load the dataset

Keep this notebook in the same GitHub repository as `scdisrupt_synth_v1.csv`.


In [ ]:
DATA_PATH = "scdisrupt_synth_v1.csv"

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head()


## 3. Basic dataset checks


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df["disruption_imminent"].value_counts().sort_index())

print("\nSource types:")
print(df["source_type"].value_counts())

print("\nTemporal splits:")
print(df["split"].value_counts())


## 4. Train/test split

The dataset already contains temporal `train`, `validation`, and `test` labels.  
This baseline trains on `train` and evaluates on `test`.


In [ ]:
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

X_train = train_df["signal_text"].astype(str)
y_train = train_df["disruption_imminent"].astype(int)

X_test = test_df["signal_text"].astype(str)
y_test = test_df["disruption_imminent"].astype(int)

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


## 5. TF-IDF + Logistic Regression baseline

This intentionally uses a simple baseline that other researchers can easily reproduce.


In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)


## 6. Evaluation metrics


In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
}

pd.DataFrame([metrics]).round(4)


In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["No imminent disruption", "Imminent disruption"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


## 7. Performance by source type


In [ ]:
rows = []

for source in sorted(test_df["source_type"].unique()):
    subset = test_df[test_df["source_type"] == source].copy()
    y_true_source = subset["disruption_imminent"].astype(int)
    y_pred_source = baseline.predict(subset["signal_text"].astype(str))

    rows.append({
        "source_type": source,
        "n": len(subset),
        "accuracy": accuracy_score(y_true_source, y_pred_source),
        "precision": precision_score(y_true_source, y_pred_source, zero_division=0),
        "recall": recall_score(y_true_source, y_pred_source, zero_division=0),
        "f1": f1_score(y_true_source, y_pred_source, zero_division=0),
    })

source_results = pd.DataFrame(rows).sort_values("f1", ascending=False)
source_results.round(4)


## 8. Positive disruption-category distribution


In [ ]:
category_counts = (
    df[df["disruption_imminent"] == 1]["disruption_category"]
    .value_counts()
    .rename_axis("disruption_category")
    .reset_index(name="records")
)

category_counts


## 9. Reproducibility and limitations

- This baseline is a reference point, not a state-of-the-art claim.
- Because the dataset is synthetic, high scores may partly reflect regularities in the generation templates.
- Future benchmarks should test harder paraphrases, domain shifts, adversarial examples, and external real-world data where licensing permits.
- Useful model comparisons include SVM, gradient boosting, BERT-family models, and multi-source fusion architectures.

## Citation

Deori, Abhilash. (2026). *SCDisrupt-Synth v1.0: A Multi-Source Synthetic Benchmark Dataset for Supply Chain Disruption Early-Warning Research*. Zenodo. https://doi.org/10.5281/zenodo.22073886
